In [37]:
import torch
print(torch.cuda.is_available())

True


In [38]:
import pandas as pd
import numpy as np

ho_df = pd.read_parquet("HOH4_merged_quotes_trades.parquet")
ho_df

,date,sym,seq,time,bsiz,bid,siz,prc,aggr,ask,asiz
0,2024-02-05,HOH4,377006,0 days 00:00:01.808937213,1,26912.0,-9223372036854775808,NaN,,26920.0,1
1,2024-02-05,HOH4,377007,0 days 00:00:01.808943921,1,26911.0,-9223372036854775808,NaN,,26920.0,1
2,2024-02-05,HOH4,377012,0 days 00:00:01.809711213,1,26911.0,-9223372036854775808,NaN,,26919.0,1
3,2024-02-05,HOH4,377019,0 days 00:00:01.858081899,1,26911.0,-9223372036854775808,NaN,,26920.0,1
4,2024-02-05,HOH4,377024,0 days 00:00:01.912609037,1,26912.0,-9223372036854775808,NaN,,26920.0,1
...,...,...,...,...,...,...,...,...,...,...,...
6155722,2024-02-16,HOH4,11918287,0 days 15:59:59.009755143,1,28075.0,-9223372036854775808,NaN,,28090.0,1
6155723,2024-02-16,HOH4,11918305,0 days 15:59:59.010284887,1,28075.0,-9223372036854775808,NaN,,28091.0,2
6155724,2024-02-16,HOH4,11918307,0 days 15:59:59.680887867,1,28075.0,-9223372036854775808,NaN,,28090.0,1
6155725,2024-02-16,HOH4,11918318,0 days 16:00:00,1,28075.0,-9223372036854775808,NaN,,28095.0,1


# Preprocessing

In [39]:
# Show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
# pd.set_option('display.max_rows', None)

In [40]:
import datetime as dt

LARGE_NEG = -9223372036854775808

# ============================================================
# 1) Preprocessing time & fill out 'N' trade type to 'B'/'S'
# ============================================================
def preprocessing_time(df: pd.DataFrame) -> pd.DataFrame:
    df_new = df.copy()
    df_new['datetime'] = df_new['date'] + df_new['time']
    df_new.drop(columns=['date', 'time'], inplace=True)

    df_new['datetime'] = pd.to_datetime(df_new['datetime'])

    mask = (
        ~df_new['datetime'].dt.date.isin([dt.date(2024, 2, 10),dt.date(2024, 2, 11)]) &
        (df_new['datetime'].dt.hour >= 7) & 
        ((df_new['datetime'].dt.hour < 13) | ((df_new['datetime'].dt.hour == 13) & (df_new['datetime'].dt.minute < 30)))
    )
    
    df_new = df_new[mask]

    is_N_trades = (df_new["aggr"] == "N")

    print(f"Number of unlabelled trades {is_N_trades.sum()}")

    # Your original heuristic: use prev quote (shift(1)) vs next record (shift(-1))
    same_dt_as_next = (df_new["datetime"] == df_new["datetime"].shift(-1))

    ask_size_decrease = (
        (df_new["asiz"].shift(1) > df_new["asiz"].shift(-1)) &
        ((df_new["asiz"].shift(1) - df_new["asiz"].shift(-1)) == df_new["siz"])
    )
    bid_size_decrease = (
        (df_new["bsiz"].shift(1) > df_new["bsiz"].shift(-1)) &
        ((df_new["bsiz"].shift(1) - df_new["bsiz"].shift(-1)) == df_new["siz"])
    )

    ask_price_up = (df_new["ask"].shift(1) < df_new["ask"].shift(-1))
    bid_price_down = (df_new["bid"].shift(1) > df_new["bid"].shift(-1))

    cond_B_trades = is_N_trades & same_dt_as_next & (ask_size_decrease | ask_price_up)
    cond_S_trades = is_N_trades & same_dt_as_next & (bid_size_decrease | bid_price_down)

    # If both fire (rare), keep as N (ambiguous)
    both = cond_B_trades & cond_S_trades
    df_new.loc[both, "aggr"] = "N"
    df_new.loc[cond_B_trades & ~both, "aggr"] = "B"
    df_new.loc[cond_S_trades & ~both, "aggr"] = "S"

    return df_new

In [41]:
ho_df_ = preprocessing_time(ho_df)
print(ho_df_.isna().sum())

Number of unlabelled trades 18434
sym               0
seq               0
bsiz              0
bid          162507
siz               0
prc         4146332
aggr              0
ask          162507
asiz              0
datetime          0
dtype: int64


In [42]:
print(f"Number of unlabelled trades after we label ones that follow our rules: {(ho_df_['aggr']=='N').sum()}")

Number of unlabelled trades after we label ones that follow our rules: 6059


In [43]:
ho_df_[ho_df_['aggr']=='N']

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime
182967,HOH4,1150026,-9223372036854775808,NaN,1,26801.0,N,NaN,-9223372036854775808,2024-02-05 07:00:27.523629293
183681,HOH4,1152712,-9223372036854775808,NaN,1,26800.0,N,NaN,-9223372036854775808,2024-02-05 07:01:00.712219997
193746,HOH4,1193564,-9223372036854775808,NaN,1,26786.0,N,NaN,-9223372036854775808,2024-02-05 07:16:02.198401979
193898,HOH4,1194244,-9223372036854775808,NaN,1,26797.0,N,NaN,-9223372036854775808,2024-02-05 07:16:09.732663849
196332,HOH4,1202749,-9223372036854775808,NaN,1,26787.0,N,NaN,-9223372036854775808,2024-02-05 07:19:11.708851717
...,...,...,...,...,...,...,...,...,...,...
6120588,HOH4,11787341,-9223372036854775808,NaN,1,28074.0,N,NaN,-9223372036854775808,2024-02-16 13:29:47.342059425
6121634,HOH4,11791030,-9223372036854775808,NaN,1,28058.0,N,NaN,-9223372036854775808,2024-02-16 13:29:55.725602469
6121635,HOH4,11791031,-9223372036854775808,NaN,1,28059.0,N,NaN,-9223372036854775808,2024-02-16 13:29:55.725602469
6122039,HOH4,11792624,-9223372036854775808,NaN,1,28059.0,N,NaN,-9223372036854775808,2024-02-16 13:29:57.415621099


In [44]:
ho_df_.loc[638043:638047]

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime


In [45]:
ho_df_.loc[625730:625734]

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime
625730,HOH4,2989676,5,27251.0,-9223372036854775808,NaN,,27254.0,2,2024-02-05 13:29:02.968154365
625731,HOH4,2989677,5,27251.0,-9223372036854775808,NaN,,27254.0,3,2024-02-05 13:29:02.968157839
625732,HOH4,2989678,3,27251.0,-9223372036854775808,NaN,,27254.0,3,2024-02-05 13:29:02.968183663
625733,HOH4,2989680,3,27251.0,-9223372036854775808,NaN,,27254.0,4,2024-02-05 13:29:02.968183841
625734,HOH4,2989682,2,27251.0,-9223372036854775808,NaN,,27254.0,4,2024-02-05 13:29:02.968249811


This seems unexpected: In most cases when trade agressor side is not labelled, we have trade happening at time t, then next row shows limit order book change at time t.

In [46]:
ho_df_.loc[643833:643840]

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime


In [47]:
ho_df_.loc[650565:650567]

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime


The following two cases are normal

In [48]:
ho_df_.loc[626437:626439]

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime
626437,HOH4,2992541,4,27250.0,-9223372036854775808,NaN,,27253.0,5,2024-02-05 13:29:10.951402629
626438,HOH4,2992549,3,27250.0,-9223372036854775808,NaN,,27253.0,5,2024-02-05 13:29:10.951683227
626439,HOH4,2992552,2,27250.0,-9223372036854775808,NaN,,27253.0,5,2024-02-05 13:29:10.958964827


In [49]:
ho_df_.loc[625666:625668]

,sym,seq,bsiz,bid,siz,prc,aggr,ask,asiz,datetime
625666,HOH4,2989462,2,27251.0,-9223372036854775808,NaN,,27254.0,1,2024-02-05 13:29:02.917243397
625667,HOH4,2989464,-9223372036854775808,NaN,1,27254.0,B,NaN,-9223372036854775808,2024-02-05 13:29:02.917245145
625668,HOH4,2989466,2,27251.0,-9223372036854775808,NaN,,27255.0,5,2024-02-05 13:29:02.917245145


In [50]:
for c in ho_df_.columns:
    print(f'column {c}')
    print("LN")
    print((ho_df_[c] == -9223372036854775808).sum())
    print("NA")
    print(ho_df_[c].isna().sum())

column sym
LN
0
NA
0
column seq
LN
0
NA
0
column bsiz
LN
162507
NA
0
column bid
LN
0
NA
162507
column siz
LN
4146332
NA
0
column prc
LN
0
NA
4146332
column aggr
LN
0
NA
0
column ask
LN
0
NA
162507
column asiz
LN
162507
NA
0
column datetime
LN
0
NA
0


In [51]:
# ============================================================
# 2) PREPROCESSING & LABELING Market Data
#    - Adds Trade_B_PxChg / Trade_B_PxSame
#    - Adds Trade_S_PxChg / Trade_S_PxSame
# ============================================================
def preprocess_market_data(df: pd.DataFrame) -> pd.DataFrame:
    print(">>> Preprocessing Data...")

    # Keep ordering stable
    sort_cols = ["datetime"] + (["seq"] if "seq" in df.columns else [])
    df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    # --- Clean INT64_MIN for relevant columns ---
    cols_to_clean = ["bsiz", "bid", "siz", "ask", "asiz"]
    cleaned = {}
    for c in cols_to_clean:
        if c not in df.columns:
            raise ValueError(f"Missing required column '{c}'")
        cleaned[c] = df[c].where(df[c] != LARGE_NEG, np.nan)

    # --- Time delta in microseconds (vectorized) ---
    dt_us = df["datetime"].diff().dt.total_seconds().fillna(0.0).to_numpy() * 1e6
    log_time = np.log1p(dt_us)

    n = len(df)
    event = np.full(n, "No_Change", dtype=object)
    log_size = np.zeros(n, dtype=np.float64)

    if n > 0:
        event[0] = "Start"
        log_size[0] = 0.0

    # --- Build "state" quotes via ffill only on quote cols ---
    quote_cols = ["bid", "bsiz", "ask", "asiz"]
    q = pd.DataFrame({c: cleaned[c] for c in quote_cols}, index=df.index).ffill()

    bid = q["bid"].to_numpy()
    ask = q["ask"].to_numpy()
    bsiz = q["bsiz"].to_numpy()
    asiz = q["asiz"].to_numpy()

    # prev values (shift by 1)
    bid_prev  = np.roll(bid, 1);  bid_prev[0]  = np.nan
    ask_prev  = np.roll(ask, 1);  ask_prev[0]  = np.nan
    bsiz_prev = np.roll(bsiz, 1); bsiz_prev[0] = np.nan
    asiz_prev = np.roll(asiz, 1); asiz_prev[0] = np.nan

    # next values (shift by -1) for "immediately after trade"
    bid_next = np.roll(bid, -1); bid_next[-1] = np.nan
    ask_next = np.roll(ask, -1); ask_next[-1] = np.nan

    bsiz_delta = bsiz - bsiz_prev
    asiz_delta = asiz - asiz_prev

    # --- Trade mask (trade rows have non-NaN siz) ---
    siz = cleaned["siz"].to_numpy()
    trade_mask = ~np.isnan(siz)

    # Side array (uppercase strings)
    side = df["aggr"].fillna("N").astype(str).str.upper().to_numpy(dtype=str)

    valid = (np.arange(n) > 0)

    # --- Quote events (same as your original) ---
    conds = [
        valid & (bid > bid_prev),
        valid & (bid < bid_prev),
        valid & (bsiz_delta > 0),
        valid & (bsiz_delta < 0),
        valid & (ask > ask_prev),
        valid & (ask < ask_prev),
        valid & (asiz_delta > 0),
        valid & (asiz_delta < 0),
    ]
    choices = [
        "Bid_Px_Up",
        "Bid_Px_Dn",
        "Bid_Sz_Up",
        "Bid_Sz_Dn",
        "Ask_Px_Up",
        "Ask_Px_Dn",
        "Ask_Sz_Up",
        "Ask_Sz_Dn",
    ]

    quote_event = np.select(conds, choices, default="No_Change")

    quote_sz = np.select(
        conds,
        [
            1.0, 1.0,
            np.abs(bsiz_delta), np.abs(bsiz_delta),
            1.0, 1.0,
            np.abs(asiz_delta), np.abs(asiz_delta),
        ],
        default=0.0
    )
    quote_log_size = np.log1p(quote_sz)

    event[:] = quote_event
    log_size[:] = quote_log_size

    # --- Override with enriched trade labels ---
    trade_apply = valid & trade_mask

    # Start from base labels
    base_trade = np.char.add("Trade_", side).astype(object)

    # Split B/S into PxChg vs PxSame using "immediately next record"
    is_B = trade_apply & (side == "B")
    is_S = trade_apply & (side == "S")

    # Your definition:
    #  - Sell: bid goes DOWN after sell
    #  - Buy:  ask goes UP after buy
    B_px_chg = is_B & (ask_next > ask_prev)
    S_px_chg = is_S & (bid_next < bid_prev)

    base_trade[B_px_chg] = "Trade_B_PxChg"
    base_trade[is_B & ~B_px_chg] = "Trade_B_PxSame"

    base_trade[S_px_chg] = "Trade_S_PxChg"
    base_trade[is_S & ~S_px_chg] = "Trade_S_PxSame"

    # (N or other labels remain Trade_N / Trade_*)
    event[trade_apply] = base_trade[trade_apply]
    log_size[trade_apply] = np.log1p(siz[trade_apply])

    # Ensure row 0 remains Start
    if n > 0:
        event[0] = "Start"
        log_size[0] = 0.0

    out = pd.DataFrame({
        "Event": event,
        "LogTime": log_time,
        "LogSize": log_size,
    })

    # Drop No_Change and Start
    out = out[~out["Event"].isin(["No_Change", "Start"])].reset_index(drop=True)
    return out


In [52]:
processed_df = preprocess_market_data(ho_df_)
print(f"\n Labels Generated:\n")
processed_df

>>> Preprocessing Data...

 Labels Generated:



,Event,LogTime,LogSize
0,Bid_Px_Up,5.807133,0.693147
1,Ask_Sz_Dn,1.375739,0.693147
2,Bid_Sz_Up,3.714742,0.693147
3,Ask_Px_Up,3.339535,0.693147
4,Bid_Sz_Up,2.051042,0.693147
...,...,...,...
4308809,Ask_Sz_Up,4.064126,0.693147
4308810,Bid_Px_Up,7.002714,0.693147
4308811,Bid_Px_Up,5.817010,0.693147
4308812,Ask_Sz_Up,9.708899,1.386294


In [53]:
processed_df.head(50)

,Event,LogTime,LogSize
0,Bid_Px_Up,5.807133,0.693147
1,Ask_Sz_Dn,1.375739,0.693147
2,Bid_Sz_Up,3.714742,0.693147
3,Ask_Px_Up,3.339535,0.693147
4,Bid_Sz_Up,2.051042,0.693147
5,Bid_Px_Up,6.373054,0.693147
6,Bid_Sz_Up,3.803368,0.693147
7,Ask_Px_Dn,9.274041,0.693147
8,Ask_Sz_Up,5.068841,0.693147
9,Bid_Sz_Dn,4.901073,0.693147


In [54]:
print(processed_df[processed_df['Event']=='Trade_S'])

Empty DataFrame
Columns: [Event, LogTime, LogSize]
Index: []


In [55]:
print(ho_df_.iloc[7:11])

         sym      seq  bsiz      bid                  siz  prc aggr      ask  asiz                      datetime
182514  HOH4  1147901     1  26801.0 -9223372036854775808  NaN       26812.0     2 2024-02-05 07:00:00.010444663
182515  HOH4  1147903     2  26801.0 -9223372036854775808  NaN       26812.0     2 2024-02-05 07:00:00.010488515
182516  HOH4  1147909     2  26801.0 -9223372036854775808  NaN       26810.0     1 2024-02-05 07:00:00.021145249
182517  HOH4  1147910     2  26801.0 -9223372036854775808  NaN       26810.0     2 2024-02-05 07:00:00.021303239


In [56]:
processed_df['Event'].value_counts()

Event
Ask_Sz_Up         715795
Bid_Sz_Up         700102
Bid_Px_Up         505810
Ask_Px_Dn         496268
Ask_Sz_Dn         459869
Bid_Sz_Dn         434341
Bid_Px_Dn         418465
Ask_Px_Up         415658
Trade_S_PxChg      56115
Trade_B_PxChg      53295
Trade_B_PxSame     24367
Trade_S_PxSame     22670
Trade_N             6059
Name: count, dtype: int64

# LSTM

In [57]:
import os
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# =========================
# CONFIGURATION
# =========================
SEQ_LEN = 100          # history length
PRED_LEN = 5          # prediction horizon (distribution over next PRED_LEN events)
BATCH_SIZE = 256      # increase for GPU / stability (adjust to fit memory)
HIDDEN_DIM = 32
EMBED_DIM = 16
LEARNING_RATE = 1e-3
EPOCHS = 10

# time-based split fractions
TRAIN_FRAC = 0.8
VAL_FRAC   = 0.1
TEST_FRAC  = 0.1

# Drop any sample window whose span contains ANY of these events
# (You asked Trade_N; I include Start/No_Change too because you mentioned dropping them earlier.
#  Remove them if you don't want that behavior.)
BANNED_EVENTS_IN_WINDOW = ["Trade_N", "Start", "No_Change"]

# Optional: also exclude these from the unconditional baseline distribution computation
# (Usually should match the window filter.)
BANNED_EVENTS_FOR_GLOBAL = ["Trade_N", "Start", "No_Change"]

warnings.filterwarnings("ignore")


# =========================
# 0. GPU / PERFORMANCE
# =========================
def seed_everything(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(123)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> Using device: {DEVICE}")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

USE_AMP = (DEVICE.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

NUM_WORKERS = min(8, os.cpu_count() or 1) if DEVICE.type == "cuda" else 0
PIN_MEMORY = (DEVICE.type == "cuda")


# =========================
# 1. DATASET
# =========================
class MarketDataset(Dataset):
    """
    Builds sliding-window samples:
      x: past SEQ_LEN events + continuous feats
      y: distribution of next PRED_LEN events
      rolling_prior: distribution of events inside x window

    Also supports dropping windows that contain banned events (e.g., Trade_N).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        seq_len: int,
        pred_len: int,
        vocab=None,
        add_unk: bool = True,
        banned_events_in_window=None,
        banned_events_for_global=None,
    ):
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)

        required = {"Event", "LogTime", "LogSize"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns in df: {missing}")

        # Ensure string
        events_str = df["Event"].astype(str)

        # vocab: build from this df if not provided, otherwise reuse
        if vocab is None:
            uniq = pd.unique(events_str).tolist()
            self.vocab = {ev: i for i, ev in enumerate(uniq)}
            if add_unk and "<UNK>" not in self.vocab:
                self.vocab["<UNK>"] = len(self.vocab)
        else:
            self.vocab = dict(vocab)

        self.idx_to_event = {i: ev for ev, i in self.vocab.items()}
        self.num_classes = len(self.vocab)
        unk_id = self.vocab.get("<UNK>", None)

        # encode events (vectorized; fast on large dataset)
        mapped = events_str.map(self.vocab)
        if mapped.isna().any():
            if unk_id is None:
                unseen = events_str[mapped.isna()].unique()[:5]
                raise KeyError(f"Unseen events in this split but no <UNK> token. Examples: {unseen}")
            mapped = mapped.fillna(unk_id)
        event_ids_np = mapped.astype(np.int64).to_numpy()

        # continuous features on time and size of quote updates/trades
        cont_np = df[["LogTime", "LogSize"]].to_numpy(dtype=np.float32)

        # store as CPU tensors (DataLoader will move batches to GPU)
        self.event_ids = torch.from_numpy(event_ids_np).long()
        self.cont_feats = torch.from_numpy(cont_np).float()

        L = len(self.event_ids)
        W = self.seq_len + self.pred_len
        if L < W:
            raise ValueError(f"Not enough rows: need at least seq_len+pred_len={W}, got {L}")

        # ---- compute valid window start indices (drop windows containing banned events) ----
        banned_events_in_window = banned_events_in_window or []
        banned_ids = [self.vocab[e] for e in banned_events_in_window if e in self.vocab]

        if len(banned_ids) > 0:
            bad_mask = np.isin(event_ids_np, np.array(banned_ids, dtype=np.int64)).astype(np.int32)
            # prefix sum trick: window sum in O(N)
            pref = np.empty(L + 1, dtype=np.int32)
            pref[0] = 0
            np.cumsum(bad_mask, out=pref[1:])  # pref[i] = sum(bad_mask[:i])
            # window sums for all starts
            window_sums = pref[W:] - pref[:-W]  # length = L-W+1
            valid = (window_sums == 0)
            self.valid_starts = np.nonzero(valid)[0].astype(np.int64)
        else:
            # all starts are valid
            self.valid_starts = np.arange(0, L - W + 1, dtype=np.int64)

        # ---- unconditional (global) baseline distribution, computed on THIS df ----
        # By default: compute overall event frequency excluding banned_events_for_global.
        banned_events_for_global = banned_events_for_global or []
        banned_global_ids = [self.vocab[e] for e in banned_events_for_global if e in self.vocab]
        if len(banned_global_ids) > 0:
            good_mask = ~np.isin(event_ids_np, np.array(banned_global_ids, dtype=np.int64))
            base_ids = event_ids_np[good_mask]
        else:
            base_ids = event_ids_np

        counts = np.bincount(base_ids, minlength=self.num_classes).astype(np.float64)
        total = counts.sum()
        if total <= 0:
            # fallback: uniform if everything got filtered (unlikely)
            self.global_dist = (np.ones(self.num_classes, dtype=np.float32) / self.num_classes)
        else:
            self.global_dist = (counts / total).astype(np.float32)

    def __len__(self):
        return int(len(self.valid_starts))

    def __getitem__(self, i: int):
        idx = int(self.valid_starts[i])

        # Input window
        x_events = self.event_ids[idx : idx + self.seq_len]         # [Seq]
        x_cont   = self.cont_feats[idx : idx + self.seq_len]        # [Seq, 2]

        # Target distribution: next pred_len events
        future_events = self.event_ids[idx + self.seq_len : idx + self.seq_len + self.pred_len]  # [Pred]
        dist_counts = torch.bincount(future_events, minlength=self.num_classes).to(torch.float32)
        dist_vector = dist_counts / dist_counts.sum().clamp_min(1.0)

        # Rolling prior: distribution inside input window
        prior_counts = torch.bincount(x_events, minlength=self.num_classes).to(torch.float32)
        rolling_prior = prior_counts / prior_counts.sum().clamp_min(1.0)

        return x_events, x_cont, dist_vector, rolling_prior


# =========================
# 2. MODEL
# =========================
class MarketHybridLSTM(nn.Module):
    def __init__(self, num_classes: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, embed_dim)
        self.feat_encoder = nn.Linear(2, 8)
        self.lstm = nn.LSTM(embed_dim + 8, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, events, continuous):
        embeds = self.embedding(events)                    # [B, Seq, Emb]
        feats  = torch.relu(self.feat_encoder(continuous)) # [B, Seq, 8]
        lstm_in = torch.cat([embeds, feats], dim=2)        # [B, Seq, Emb+8]
        out, _ = self.lstm(lstm_in)
        last_hidden = out[:, -1, :]                        # [B, H]
        logits = self.fc(last_hidden)                      # [B, C]
        return self.log_softmax(logits)                    # [B, C] log-probs


# =========================
# 3. EVALUATION
# =========================
@torch.no_grad()
def evaluate(model, loader, criterion, global_log_prior):
    model.eval()
    sum_model = 0.0
    sum_roll  = 0.0
    sum_glob  = 0.0
    n = 0

    for x_ev, x_cont, y_target, y_rolling_prior in loader:
        bs = x_ev.size(0)
        x_ev = x_ev.to(DEVICE, non_blocking=True)
        x_cont = x_cont.to(DEVICE, non_blocking=True)
        y_target = y_target.to(DEVICE, non_blocking=True)
        y_rolling_prior = y_rolling_prior.to(DEVICE, non_blocking=True)

        log_probs = model(x_ev, x_cont)  # [B, C]

        loss_model = criterion(log_probs, y_target)
        roll_log = torch.log(y_rolling_prior + 1e-9)
        loss_roll = criterion(roll_log, y_target)

        glob_log = global_log_prior.expand_as(log_probs)
        loss_glob = criterion(glob_log, y_target)

        # batchmean -> average per batch; weight by bs for dataset-level average
        sum_model += float(loss_model.item()) * bs
        sum_roll  += float(loss_roll.item())  * bs
        sum_glob  += float(loss_glob.item())  * bs
        n += bs

    return sum_model / n, sum_roll / n, sum_glob / n


# =========================
# 4. TRAINING
# =========================
def run_training(processed_df: pd.DataFrame):
    # Basic sanity
    assert all(c in processed_df.columns for c in ["Event", "LogTime", "LogSize"]), \
        "processed_df must contain columns: Event, LogTime, LogSize"

    # Ensure time order (YOU should already have this sorted; this is a safeguard)
    # If you have a datetime column, sort by it before calling this function.
    processed_df = processed_df.reset_index(drop=True)

    # time-based contiguous split
    N = len(processed_df)
    train_end = int(TRAIN_FRAC * N)
    val_end   = int((TRAIN_FRAC + VAL_FRAC) * N)

    train_df = processed_df.iloc[:train_end].reset_index(drop=True)
    val_df   = processed_df.iloc[train_end:val_end].reset_index(drop=True)
    test_df  = processed_df.iloc[val_end:].reset_index(drop=True)

    print(f">>> Split sizes (rows): train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    # Build vocab from TRAIN only
    train_dataset = MarketDataset(
        train_df, SEQ_LEN, PRED_LEN,
        vocab=None,
        add_unk=True,
        banned_events_in_window=BANNED_EVENTS_IN_WINDOW,
        banned_events_for_global=BANNED_EVENTS_FOR_GLOBAL,
    )
    vocab = train_dataset.vocab

    # Reuse vocab for val/test, apply same window filtering
    val_dataset = MarketDataset(
        val_df, SEQ_LEN, PRED_LEN,
        vocab=vocab,
        add_unk=True,
        banned_events_in_window=BANNED_EVENTS_IN_WINDOW,
        banned_events_for_global=BANNED_EVENTS_FOR_GLOBAL,
    )
    test_dataset = MarketDataset(
        test_df, SEQ_LEN, PRED_LEN,
        vocab=vocab,
        add_unk=True,
        banned_events_in_window=BANNED_EVENTS_IN_WINDOW,
        banned_events_for_global=BANNED_EVENTS_FOR_GLOBAL,
    )

    print(f">>> Samples after dropping banned windows:")
    print(f"    train samples={len(train_dataset)}, val samples={len(val_dataset)}, test samples={len(test_dataset)}")

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,   # good for optimization; each sample is still a causal window
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )

    # Model
    model = MarketHybridLSTM(train_dataset.num_classes, EMBED_DIM, HIDDEN_DIM).to(DEVICE)

    # Optional: torch.compile (PyTorch 2.x)
    try:
        if hasattr(torch, "compile"):
            model = torch.compile(model)
    except Exception as e:
        print(f">>> torch.compile unavailable or failed (continuing without): {e}")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # KLDivLoss expects input=log-probs and target=probs
    criterion = nn.KLDivLoss(reduction="batchmean")

    # Unconditional baseline (TRAIN overall event frequencies, excluding banned global events)
    global_prior = torch.tensor(train_dataset.global_dist, dtype=torch.float32, device=DEVICE)
    global_log_prior = torch.log(global_prior + 1e-9).unsqueeze(0)  # [1, C]

    # Train with best-val checkpoint
    best_val = float("inf")
    best_state = None

    print(f"\n>>> Starting Training (classes={train_dataset.num_classes})")
    print(f"Predict next {PRED_LEN} event distribution from past {SEQ_LEN}.\n")
    print(f"Banned events in any sample window: {BANNED_EVENTS_IN_WINDOW}\n")

    for epoch in range(EPOCHS):
        model.train()
        sum_train = 0.0
        n_train = 0

        for x_ev, x_cont, y_target, _y_roll in train_loader:
            bs = x_ev.size(0)

            x_ev = x_ev.to(DEVICE, non_blocking=True)
            x_cont = x_cont.to(DEVICE, non_blocking=True)
            y_target = y_target.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                log_probs = model(x_ev, x_cont)
                loss = criterion(log_probs, y_target)

            scaler.scale(loss).backward()
            # Optional: gradient clipping can help LSTM stability
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            sum_train += float(loss.item()) * bs
            n_train += bs

        train_kl = sum_train / max(1, n_train)
        val_model, val_roll, val_glob = evaluate(model, val_loader, criterion, global_log_prior)

        # positive gain means model beats baseline
        gain_vs_roll = val_roll - val_model
        gain_vs_glob = val_glob - val_model

        print(
            f"Epoch {epoch+1:02d}/{EPOCHS} | "
            f"Train KL: {train_kl:.4f} | "
            f"Val KL (Model/Roll/Global): {val_model:.4f} / {val_roll:.4f} / {val_glob:.4f} | "
            f"Gain (vs Roll/Global): {gain_vs_roll:.4f} / {gain_vs_glob:.4f}"
        )

        if val_model < best_val:
            best_val = val_model
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Test with best model
    if best_state is not None:
        model.load_state_dict(best_state)

    test_model, test_roll, test_glob = evaluate(model, test_loader, criterion, global_log_prior)
    print("\n>>> TEST RESULTS")
    print(f"TEST KL (Model/Roll/Global): {test_model:.4f} / {test_roll:.4f} / {test_glob:.4f}")
    print(f"TEST Gain (vs Roll/Global): {test_roll - test_model:.4f} / {test_glob - test_model:.4f}")

    return model, test_loader, test_dataset


# 5. RUN experiments
if __name__ == "__main__":
    assert "processed_df" in globals(), "Please define processed_df before running this script."
    trained_model, test_loader, test_dataset = run_training(processed_df)


>>> Using device: cuda
>>> Split sizes (rows): train=3447051, val=430881, test=430882
>>> Samples after dropping banned windows:
    train samples=3161754, val samples=384788, test samples=376489

>>> Starting Training (classes=14)
Predict next 1 event distribution from past 100.

Banned events in any sample window: ['Trade_N', 'Start', 'No_Change']

Epoch 01/10 | Train KL: 1.7488 | Val KL (Model/Roll/Global): 1.7347 / 2.3509 / 2.1747 | Gain (vs Roll/Global): 0.6162 / 0.4400
Epoch 02/10 | Train KL: 1.7071 | Val KL (Model/Roll/Global): 1.7212 / 2.3509 / 2.1747 | Gain (vs Roll/Global): 0.6298 / 0.4535
Epoch 03/10 | Train KL: 1.6984 | Val KL (Model/Roll/Global): 1.7166 / 2.3509 / 2.1747 | Gain (vs Roll/Global): 0.6343 / 0.4581
Epoch 04/10 | Train KL: 1.6937 | Val KL (Model/Roll/Global): 1.7130 / 2.3509 / 2.1747 | Gain (vs Roll/Global): 0.6379 / 0.4617
Epoch 05/10 | Train KL: 1.6906 | Val KL (Model/Roll/Global): 1.7106 / 2.3509 / 2.1747 | Gain (vs Roll/Global): 0.6403 / 0.4641
Epoch 06/10 

In [58]:
def run_training(processed_df: pd.DataFrame):
    # Basic sanity
    assert all(c in processed_df.columns for c in ["Event", "LogTime", "LogSize"]), \
        "processed_df must contain columns: Event, LogTime, LogSize"

    # Ensure time order (YOU should already have this sorted; this is a safeguard)
    # If you have a datetime column, sort by it before calling this function.
    processed_df = processed_df.reset_index(drop=True)

    # time-based contiguous split
    N = len(processed_df)
    train_end = int(TRAIN_FRAC * N)
    val_end   = int((TRAIN_FRAC + VAL_FRAC) * N)

    train_df = processed_df.iloc[:train_end].reset_index(drop=True)
    val_df   = processed_df.iloc[train_end:val_end].reset_index(drop=True)
    test_df  = processed_df.iloc[val_end:].reset_index(drop=True)

    print(f">>> Split sizes (rows): train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    # Build vocab from TRAIN only
    train_dataset = MarketDataset(
        train_df, SEQ_LEN, PRED_LEN,
        vocab=None,
        add_unk=True,
        banned_events_in_window=BANNED_EVENTS_IN_WINDOW,
        banned_events_for_global=BANNED_EVENTS_FOR_GLOBAL,
    )
    vocab = train_dataset.vocab

    # Reuse vocab for val/test, apply same window filtering
    val_dataset = MarketDataset(
        val_df, SEQ_LEN, PRED_LEN,
        vocab=vocab,
        add_unk=True,
        banned_events_in_window=BANNED_EVENTS_IN_WINDOW,
        banned_events_for_global=BANNED_EVENTS_FOR_GLOBAL,
    )
    test_dataset = MarketDataset(
        test_df, SEQ_LEN, PRED_LEN,
        vocab=vocab,
        add_unk=True,
        banned_events_in_window=BANNED_EVENTS_IN_WINDOW,
        banned_events_for_global=BANNED_EVENTS_FOR_GLOBAL,
    )

    print(f">>> Samples after dropping banned windows:")
    print(f"    train samples={len(train_dataset)}, val samples={len(val_dataset)}, test samples={len(test_dataset)}")

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,   # good for optimization; each sample is still a causal window
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )

    return test_loader, test_dataset

test_loader, test_dataset = run_training(processed_df)



>>> Split sizes (rows): train=3447051, val=430881, test=430882
>>> Samples after dropping banned windows:
    train samples=3161754, val samples=384788, test samples=376489


In [59]:
@torch.no_grad()
def collect_all_predictions(model, loader, dataset):
    model.eval()

    all_true = []
    all_pred = []
    all_prob_true = []
    all_prob_pred = []

    for x_ev, x_cont, y_target, _ in loader:
        x_ev = x_ev.to(DEVICE)
        x_cont = x_cont.to(DEVICE)
        y_target = y_target.to(DEVICE)

        log_probs = model(x_ev, x_cont)
        probs = torch.exp(log_probs)

        pred_idx = torch.argmax(probs, dim=1)
        true_idx = torch.argmax(y_target, dim=1)

        for i in range(x_ev.size(0)):
            true_event = dataset.idx_to_event[int(true_idx[i])]
            pred_event = dataset.idx_to_event[int(pred_idx[i])]
            prob_true  = probs[i, true_idx[i]].item()
            prob_pred  = probs[i, pred_idx[i]].item()

            all_true.append(true_event)
            all_pred.append(pred_event)
            all_prob_true.append(prob_true)
            all_prob_pred.append(prob_pred)

    df = pd.DataFrame({
        "True": all_true,
        "Predicted": all_pred,
        "Prob_True": all_prob_true,
        "Prob_Pred": all_prob_pred
    })

    return df


In [60]:
pred_df = collect_all_predictions(trained_model, test_loader, test_dataset)

In [61]:
pred_df.head(50)

,True,Predicted,Prob_True,Prob_Pred
0,Ask_Px_Dn,Ask_Px_Dn,0.245142,0.245142
1,Bid_Px_Dn,Ask_Sz_Up,0.062922,0.228689
2,Bid_Sz_Dn,Bid_Px_Up,0.159406,0.231270
3,Bid_Sz_Dn,Ask_Sz_Up,0.173240,0.235669
4,Ask_Px_Dn,Ask_Sz_Up,0.116043,0.240154
5,Bid_Px_Up,Ask_Sz_Up,0.092162,0.268030
6,Bid_Px_Dn,Bid_Px_Dn,0.333140,0.333140
7,Bid_Px_Dn,Bid_Px_Up,0.106212,0.261918
8,Bid_Sz_Dn,Bid_Px_Up,0.192906,0.282326
9,Bid_Px_Dn,Bid_Sz_Dn,0.164390,0.191768


In [62]:
pred_df['True'].value_counts()

True
Ask_Sz_Up         63053
Bid_Sz_Up         62402
Bid_Px_Up         45799
Ask_Px_Dn         41741
Ask_Sz_Dn         40062
Bid_Sz_Dn         39620
Bid_Px_Dn         38086
Ask_Px_Up         35467
Trade_B_PxChg      3655
Trade_S_PxChg      3500
Trade_B_PxSame     1824
Trade_S_PxSame     1280
Name: count, dtype: int64

In [63]:
pred_df['Predicted'].value_counts()

Predicted
Bid_Sz_Up         71703
Ask_Sz_Up         64657
Bid_Sz_Dn         50010
Bid_Px_Up         49253
Ask_Sz_Dn         45093
Ask_Px_Dn         41340
Bid_Px_Dn         26130
Ask_Px_Up         25729
Trade_S_PxChg       981
Trade_B_PxChg       972
Trade_S_PxSame      317
Trade_B_PxSame      304
Name: count, dtype: int64

In [64]:
pred_df[pred_df['Predicted']=='Trade_B'].head(20)

,True,Predicted,Prob_True,Prob_Pred


In [65]:
pred_df.loc[3420:3427]

,True,Predicted,Prob_True,Prob_Pred
3420,Ask_Sz_Up,Ask_Sz_Up,0.505196,0.505196
3421,Ask_Sz_Up,Ask_Sz_Up,0.495540,0.495540
3422,Ask_Px_Dn,Ask_Sz_Up,0.122896,0.426966
3423,Ask_Px_Up,Ask_Sz_Up,0.138904,0.453759
3424,Ask_Sz_Up,Ask_Px_Dn,0.230095,0.409701
3425,Ask_Px_Dn,Ask_Px_Dn,0.455867,0.455867
3426,Ask_Sz_Up,Ask_Sz_Up,0.487550,0.487550
3427,Ask_Px_Dn,Ask_Sz_Up,0.088455,0.496582


In [66]:
# ============================================================
# Trade-direction evaluation vs "last-trade sign" baseline
#   - Evaluates ONLY when the target next event is a trade (B/S)
#   - Collapses model probs over Trade_B* and Trade_S* into p(B), p(S)
#   - Baseline: predict next trade sign = last trade sign seen in x window
#     (fallback to unconditional trade-direction prior if no prior trade in window)
# ============================================================

import math
import numpy as np
import pandas as pd
import torch


def _get_trade_sign_id_sets(dataset):
    """
    Buy-trade classes: any event string starting with 'Trade_B'
    Sell-trade classes: any event string starting with 'Trade_S'
    Works for: Trade_B, Trade_S, Trade_B_PxChg, Trade_B_PxSame, Trade_Sell, etc.
    """
    buy_ids, sell_ids = [], []
    for ev, idx in dataset.vocab.items():
        if isinstance(ev, str):
            if ev.startswith("Trade_B"):
                buy_ids.append(idx)
            elif ev.startswith("Trade_S"):
                sell_ids.append(idx)
    return buy_ids, sell_ids


def _trade_direction_prior_from_global_dist(dataset, buy_ids, sell_ids, eps=1e-12):
    """
    Computes P(B|trade), P(S|trade) from dataset.global_dist if available,
    otherwise falls back to 0.5/0.5.
    """
    if hasattr(dataset, "global_dist") and dataset.global_dist is not None:
        g = np.asarray(dataset.global_dist, dtype=np.float64)
        pB = float(g[buy_ids].sum()) if len(buy_ids) else 0.0
        pS = float(g[sell_ids].sum()) if len(sell_ids) else 0.0
        denom = pB + pS
        if denom > eps:
            return pB / denom, pS / denom
    return 0.5, 0.5


@torch.no_grad()
def evaluate_trade_sign_vs_last_trade(
    model,
    loader,
    dataset,
    eps: float = 1e-9,
    return_df: bool = True,
):
    """
    Returns:
      summary dict + (optional) per-sample DataFrame.

    Metrics computed on TRADE-ONLY subset:
      - model_logloss: binary log loss on trade direction (B vs S)
      - base_logloss:  binary log loss for baseline (copy last trade sign)
      - info_gain:     base_logloss - model_logloss  (positive is good)
      - model_acc / base_acc on trade direction
      - n_trade_targets: number of evaluated samples
      - frac_no_prior_trade: fraction where baseline had to use unconditional prior
    """
    model.eval()

    buy_ids, sell_ids = _get_trade_sign_id_sets(dataset)
    if len(buy_ids) == 0 or len(sell_ids) == 0:
        raise ValueError(
            f"Could not find both buy and sell trade classes in vocab. "
            f"Found |buy_ids|={len(buy_ids)}, |sell_ids|={len(sell_ids)}. "
            f"Make sure your events are named like 'Trade_B*' and 'Trade_S*'."
        )

    buy_ids_t = torch.tensor(buy_ids, dtype=torch.long, device=DEVICE)
    sell_ids_t = torch.tensor(sell_ids, dtype=torch.long, device=DEVICE)

    # Baseline fallback: unconditional P(B|trade), P(S|trade)
    pB_prior, pS_prior = _trade_direction_prior_from_global_dist(dataset, buy_ids, sell_ids)
    pB_prior = float(np.clip(pB_prior, eps, 1.0 - eps))
    pS_prior = float(np.clip(pS_prior, eps, 1.0 - eps))

    sum_model_ll = 0.0
    sum_base_ll = 0.0
    sum_model_acc = 0
    sum_base_acc = 0
    n_eval = 0
    n_no_prior = 0

    rows = [] if return_df else None

    for x_ev, x_cont, y_target, _ in loader:
        x_ev = x_ev.to(DEVICE, non_blocking=True)         # [B, Seq]
        x_cont = x_cont.to(DEVICE, non_blocking=True)     # [B, Seq, d]
        y_target = y_target.to(DEVICE, non_blocking=True) # [B, C] (one-hot if PRED_LEN=1)

        log_probs = model(x_ev, x_cont)                   # [B, C]
        probs = torch.exp(log_probs)                      # [B, C]

        # Collapse to trade-direction probs for model: p(B|trade), p(S|trade)
        pB_model_raw = probs.index_select(1, buy_ids_t).sum(dim=1)    # [B]
        pS_model_raw = probs.index_select(1, sell_ids_t).sum(dim=1)   # [B]
        pTrade = (pB_model_raw + pS_model_raw).clamp_min(eps)
        pB_model = (pB_model_raw / pTrade).clamp(eps, 1.0 - eps)      # [B]

        # Determine if target is a trade (buy or sell), using y_target distribution
        yB = y_target.index_select(1, buy_ids_t).sum(dim=1)           # [B]
        yS = y_target.index_select(1, sell_ids_t).sum(dim=1)          # [B]
        is_trade_target = (yB + yS) > 0.0                             # [B]

        if not is_trade_target.any():
            continue

        # Convert to hard label for accuracy (PRED_LEN=1 => one-hot)
        # true_buy = 1 if buy trade, else 0 (sell trade)
        true_buy = (yB > yS).to(torch.float32)                        # [B], only meaningful where is_trade_target

        # Baseline p(B): copy last trade sign within x window; fallback to unconditional prior if none.
        x_ev_cpu = x_ev.detach().cpu().numpy()                        # [B, Seq]
        Bset = set(buy_ids)
        Sset = set(sell_ids)

        pB_base_list = []
        last_sign_list = []

        for seq in x_ev_cpu:
            last = None
            # scan backwards to find last trade event in history
            for eid in seq[::-1]:
                if int(eid) in Bset:
                    last = "B"
                    break
                if int(eid) in Sset:
                    last = "S"
                    break

            if last is None:
                n_no_prior += 1
                pB_base_list.append(pB_prior)
                last_sign_list.append("None")
            elif last == "B":
                pB_base_list.append(1.0 - eps)
                last_sign_list.append("B")
            else:
                pB_base_list.append(eps)
                last_sign_list.append("S")

        pB_base = torch.tensor(pB_base_list, dtype=torch.float32, device=DEVICE).clamp(eps, 1.0 - eps)

        # Now score ONLY trade-target samples in this batch
        idxs = torch.nonzero(is_trade_target, as_tuple=False).squeeze(1)

        tb = true_buy.index_select(0, idxs)            # [K]
        pm = pB_model.index_select(0, idxs)            # [K]
        pb = pB_base.index_select(0, idxs)             # [K]

        # binary log loss (cross-entropy) with hard labels
        # loss = -[ y log p + (1-y) log (1-p) ]
        model_ll = -(tb * torch.log(pm) + (1.0 - tb) * torch.log(1.0 - pm))
        base_ll  = -(tb * torch.log(pb) + (1.0 - tb) * torch.log(1.0 - pb))

        sum_model_ll += float(model_ll.sum().item())
        sum_base_ll  += float(base_ll.sum().item())

        # accuracy
        pred_model = (pm >= 0.5).to(torch.float32)
        pred_base  = (pb >= 0.5).to(torch.float32)

        sum_model_acc += int((pred_model == tb).sum().item())
        sum_base_acc  += int((pred_base == tb).sum().item())

        n_eval += int(tbsz := tb.numel())

        if return_df:
            # For interpretability, also attach true/pred event strings using argmax over full class space
            true_idx_full = torch.argmax(y_target, dim=1).detach().cpu().numpy()
            pred_idx_full = torch.argmax(probs, dim=1).detach().cpu().numpy()

            pm_cpu = pm.detach().cpu().numpy()
            pb_cpu = pb.detach().cpu().numpy()
            tb_cpu = tb.detach().cpu().numpy().astype(int)

            idxs_cpu = idxs.detach().cpu().numpy()
            for j, bi in enumerate(idxs_cpu):
                rows.append({
                    "True_Event": dataset.idx_to_event[int(true_idx_full[bi])],
                    "Pred_Event": dataset.idx_to_event[int(pred_idx_full[bi])],
                    "True_Sign": "B" if tb_cpu[j] == 1 else "S",
                    "Model_pB": float(pm_cpu[j]),
                    "Base_pB": float(pb_cpu[j]),
                    "Last_Trade_Sign_in_Window": last_sign_list[int(bi)],
                    "Model_LogLoss": float(model_ll[j].item()),
                    "Base_LogLoss": float(base_ll[j].item()),
                })


    if n_eval == 0:
        raise RuntimeError("No trade-target samples found in this loader (targets never Trade_B*/Trade_S*).")

    model_logloss = sum_model_ll / n_eval
    base_logloss  = sum_base_ll / n_eval
    info_gain     = base_logloss - model_logloss
    model_acc     = sum_model_acc / n_eval
    base_acc      = sum_base_acc / n_eval
    frac_no_prior = n_no_prior / max(1, n_eval)  # counted per-sample when building baseline, close enough

    summary = {
        "n_trade_targets": int(n_eval),
        "model_logloss": float(model_logloss),
        "baseline_logloss": float(base_logloss),
        "info_gain (baseline - model)": float(info_gain),
        "model_acc": float(model_acc),
        "baseline_acc": float(base_acc),
        "fallback_prior_pB": float(pB_prior),
        "frac_no_prior_trade_in_window": float(frac_no_prior),
    }

    out_df = pd.DataFrame(rows) if return_df else None
    return summary, out_df


# =========================
# USAGE (after you have trained_model, test_loader, test_dataset)
# =========================
trade_sign_summary, trade_sign_df = evaluate_trade_sign_vs_last_trade(
    trained_model, test_loader, test_dataset, return_df=True
)

print("\n>>> Trade-direction eval vs last-trade-sign baseline")
for k, v in trade_sign_summary.items():
    if isinstance(v, float):
        print(f"{k}: {v:.6f}")
    else:
        print(f"{k}: {v}")

# Optional: inspect hardest cases where model underperforms baseline
if trade_sign_df is not None and len(trade_sign_df) > 0:
    print("\n>>> Examples where baseline beats model most (by logloss)")
    print(
        trade_sign_df.assign(delta=lambda d: d["Model_LogLoss"] - d["Base_LogLoss"])
        .sort_values("delta", ascending=False)
        .head(10)
        .to_string(index=False)
    )



>>> Trade-direction eval vs last-trade-sign baseline
n_trade_targets: 10259
model_logloss: 0.456311
baseline_logloss: nan
info_gain (baseline - model): nan
model_acc: 0.773955
baseline_acc: 0.677454
fallback_prior_pB: 0.527906
frac_no_prior_trade_in_window: 7.536797

>>> Examples where baseline beats model most (by logloss)
    True_Event Pred_Event True_Sign  Model_pB      Base_pB Last_Trade_Sign_in_Window  Model_LogLoss  Base_LogLoss    delta
Trade_S_PxSame  Bid_Sz_Up         S  0.961429 1.000000e-09                         S       3.255259     -0.000000 3.255259
Trade_S_PxSame  Bid_Sz_Up         S  0.955720 1.000000e-09                         S       3.117226     -0.000000 3.117226
Trade_S_PxSame  Bid_Sz_Up         S  0.915409 1.000000e-09                         S       2.469923     -0.000000 2.469923
Trade_S_PxSame  Bid_Sz_Up         S  0.911386 1.000000e-09                         S       2.423463     -0.000000 2.423463
Trade_S_PxSame  Bid_Sz_Up         S  0.902149 1.000000e-09